# Planted truth on the human vs mouse data (notebook 103)

**The question.** How often does each calling method find a real pathway-level difference of a known kind, and how
many sets does it call that were not planted? Known differences are planted into our AKI data under the relabeling
that splits both IR and both control mice across the two groups (injury cancels; it is the model's nuisance). Each
method then runs as notebook 71 runs it: the positional F-test, the segment F-test and clustering.

**The frozen protocol** is `docs/results/pt-planted-benchmark.md` (commit a8f55de): 15 sets per replicate, 60% of
members, four shapes (shape, located, level, bump) at A = 0.5 and 1 log2, 8 replicates each; planted genes are
refitted and spliced into the cached baseline.

**Sections.** 1 setup, inputs and guards; 2 baselines; 3 planting runs; 4 summaries and readings; 5 figures;
6 reading and run record.

## 1 · Setup, inputs and guards

In [ ]:
import argparse
import json
import os
import sys
import time
import warnings
from dataclasses import replace
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
package_root = results_root / 'package_pseudospace'
T71 = results_root / 'pt_pathways_package_pseudospace'
output = results_root / 'pt_planted_benchmark_hvm'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '103.planted_hvm.1'
SEED = 103
DS = 'human_vs_mouse'
FINGERPRINTS = {'hvm': 'efafc9ac2e99', 'aki': 'bb4a80c1b484'}
BASE, NULL_LABEL = 'swap_HUK1_COR1_Ctrl1A4', 'swap_HUK1_COR1_Ctrl1A2'       # file and cache slugs
BASE_P, NULL_P = 'swap:HUK1_COR1+Ctrl1A4', 'swap:HUK1_COR1+Ctrl1A2'           # partition labels
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
ALPHA, CAMERA_COR, DF1, DF2 = .05, .01, 6, 6
SET_SIZES = (10, 300)
K_SETS, PLANT_SIZES, MAX_JACCARD, FRAC, MIN_PLANTED = 15, (20, 150), .1, .6, 10
SCENARIOS, AMPLITUDES, N_REP = ('shape', 'located', 'level', 'bump'), (.5, 1.), 8
DETECT_SENS, DETECT_FALSE, TIE = .5, 2., .1
ZONE_MARKERS = {'PT-S1': ['Slc5a12'], 'PT-S2': ['Slc13a3'], 'PT-S3': ['Slc16a9']}
ZONE_SHORT = {'PT-S1': 'S1', 'PT-S2': 'S2', 'PT-S3': 'S3'}
NODE_SPACING = 1 / 29
EXPRESSION_COLUMN = 'expression_per_10k'
METHODS = ['positional F-test', 'segment F-test', 'clustering']
REQUIRED = [package_root / 'hvm' / 'pt_units.h5ad', package_root / 'aki' / 'pt_units.h5ad',
            results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv',
            T71 / 'tables' / f'gene_statistics_{DS}.csv', T71 / 'tables' / f'gene_sets_{DS}.json',
            T71 / 'tables' / f'universe_{DS}.csv', T71 / 'tables' / 'method_b_zones.csv',
            T71 / 'tables' / f'pathway_tests_{DS}_cameraPR_{BASE}.csv']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70 and 71 first)')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pseudospace.discrete_comparators import segment_step_lr
from pseudospace.package_inputs import aki_inputs, hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import (GeneModel, camera_pr, clustering_first_inputs, moderated_f, partition_groups,
                                           single_split_design, specimen_shape_lr, standard_gene_model)
from pseudospace.planting import choose_sets, effect_shape, plant_counts
from pseudospace.stage_cache import cached_payload, code_digest

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
METHOD_COLOR = {'positional F-test': '#1b8a5a', 'segment F-test': '#5fb88a', 'clustering': '#eda100'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
orth = pd.read_csv(results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv')
MT = sorted(g for g in orth.mouse_symbol.astype(str) if g.startswith('mt-'))
U = {k: read_pt_units(package_root / k / 'pt_units.h5ad', k) for k in ('hvm',)}
for k, u in U.items():
    assert u.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINTS[k], k
D = hvm_inputs(U['hvm'], data_root, orth, exclude_genes=MT)
del U
GENES = list(D['genes'])
SETS = D['gene_sets']
assert GENES == pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv').gene.astype(str).tolist(), 'G1: universe'
assert SETS == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G1: sets'
COUNTS = D['counts'].tocsr()
POS, SPEC, LIB = np.asarray(D['position'], float), np.asarray(D['specimen']).astype(str), np.asarray(D['library'], float)
SEG = np.asarray(D['segment']).astype(str)
PARTS = partition_groups(SPEC, np.asarray(D['case'], bool))
assert list(PARTS)[1:] == [NULL_P, BASE_P], list(PARTS)
GROUP, NUIS = PARTS[BASE_P]
GROUP1 = sorted(set(SPEC[GROUP == 1]))
assert GROUP1 == ['Ctrl1A4', 'HUK1_COR1'], GROUP1
LOG_MEAN = np.log(np.asarray(COUNTS.mean(axis=0)).ravel())
guard_log.append(f"G1: universe ({len(GENES):,} genes) and sets ({len(SETS):,}) equal notebook 71's; base partition "
                 f"{BASE} (group 1 = {', '.join(GROUP1)}; nuisance = the condition)")
print('\n'.join('· ' + g for g in guard_log))

## 2 · Baselines

- Positional F-test: 71's cached gene model and statistics for the base partition; `moderated_f` and `camera_pr`
  must reproduce 71's z and calls (G2).
- Segment F-test: `segment_step_lr` on every gene under the cached dispersion, fitted once here.
- Clustering: 71's `run_b` on the cached model, with the other relabeling as the fair null; it must reproduce 71's
  cached calls (G3).

In [ ]:
def stored(folder, stage):
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


BASE_MODEL = model_from(stored(T71, f'gene_model_{DS}_{BASE}'), GENES)
table71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)['relabeling 2']
assert np.allclose(table71.T_spatial.to_numpy(), BASE_MODEL.statistics.T_spatial.to_numpy(), rtol=1e-12, atol=0)
BASE_T, BASE_LR = table71.T_spatial.to_numpy(float), table71.LR_spec.to_numpy(float)
ALPHA_ALL = np.asarray(BASE_MODEL.dispersion['alpha'], float)


def f_calls(T, LR):
    z = moderated_f(pd.Series(T, index=GENES), DF1, LR, DF2, covariate=LOG_MEAN, robust=False).z
    return set(camera_pr(z, SETS, inter_gene_cor=CAMERA_COR).query('call').pathway_id), z


BASE_CALLS = {}
BASE_CALLS['positional F-test'], z0 = f_calls(BASE_T, BASE_LR)
np.testing.assert_allclose(z0.to_numpy(), table71.z.to_numpy(), rtol=1e-9, atol=1e-9)
cam71 = pd.read_csv(T71 / 'tables' / f'pathway_tests_{DS}_cameraPR_{BASE}.csv')
assert BASE_CALLS['positional F-test'] == set(cam71.pathway_id[cam71.call.astype(bool)]), 'G2: F-test calls'
guard_log.append(f"G2: the cached model reproduces 71's z and its {len(BASE_CALLS['positional F-test'])} F-test calls on the base partition")

t0 = time.perf_counter()
seg_payload = cached_payload('segment_baseline', lambda: {k: np.asarray(v) for k, v in segment_step_lr(
    COUNTS, LIB, SEG, GROUP, SPEC, ALPHA_ALL, nuisance=NUIS).items() if k in ('T_step', 'LR_spec', 'df1', 'df2')},
    root=cache, params={'logic': NOTEBOOK_LOGIC_VERSION, 'partition': BASE}, inputs={'genes': len(GENES)},
    code=segment_step_lr)
SEG_T, SEG_LR = seg_payload['T_step'].astype(float), seg_payload['LR_spec'].astype(float)
SEG_DF1, SEG_DF2 = int(seg_payload['df1']), int(seg_payload['df2'])


def s_calls(T, LR):
    z = moderated_f(pd.Series(T, index=GENES), SEG_DF1, LR, SEG_DF2, covariate=LOG_MEAN, robust=False).z
    return set(camera_pr(z, SETS, inter_gene_cor=CAMERA_COR).query('call').pathway_id)


BASE_CALLS['segment F-test'] = s_calls(SEG_T, SEG_LR)
print(f'Segment baseline {time.perf_counter() - t0:.0f} s; df1 {SEG_DF1}, df2 {SEG_DF2}')

import pseudospace_reconstruction as prc  # noqa: E402
from pseudospace_reconstruction.compare import CompareParams, Design, compare_groups  # noqa: E402
from pseudospace_reconstruction.compare.pathways import PathwayParams, run_pathways  # noqa: E402
from pseudospace_reconstruction.io.gene_sets import build_gene_sets  # noqa: E402

COMPARE_PARAMS = CompareParams(prior_k=0.)
PATHWAY_PARAMS = PathwayParams(min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
EXT_SETS = build_gene_sets({lib: {p.split('::', 1)[1]: v for p, v in SETS.items() if p.startswith(lib + '::')}
                            for lib in LIBRARIES}, GENES, min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
zones = pd.read_csv(T71 / 'tables' / 'method_b_zones.csv').set_index('dataset').loc['Human vs mouse PT']
ZONES = np.array([zones['S1|S2 cut'], zones['S2|S3 cut']], float)
per = pd.Series(GROUP, index=SPEC).groupby(level=0).first()
DESIGN = single_split_design(Design.from_labels(list(per.index), list(np.where(per.eq(1), 'case', 'reference')),
                                                case_label='case', reference_label='reference'),
                             sorted(set(SPEC[PARTS[NULL_P][0] == 1])))


def clustering_calls(model):
    axis, fit = clustering_first_inputs(model, POS, GROUP, SPEC, LIB, zone_names=list(ZONE_MARKERS), zone_cuts=ZONES,
                                        zone_short_names=ZONE_SHORT)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        result = compare_groups(fit, axis, DESIGN, node_spacing=NODE_SPACING, elsewhere_flags=None, params=COMPARE_PARAMS)
        gt = result.gene_table
        curves = result.curves()
        pathways = run_pathways(curves['D'].astype(float), curves['P'].astype(float), curves['N'].astype(float),
                                curves['weights'], curves['grid'], EXT_SETS, gt[EXPRESSION_COLUMN].to_numpy(),
                                gt['group'].to_numpy(), cluster_labels=gt['cluster'].to_numpy(), flagged=None,
                                groups=list(result.shape_groups.index), pair_names=[str(n) for n in curves['pair_names']],
                                split_names=[str(n) for n in curves['null_names']], pair_samples=DESIGN.pairs,
                                params=PATHWAY_PARAMS)
    ora = pd.concat([pathways.by_group, pathways.by_cluster], ignore_index=True)
    return set((ora.library + '::' + ora.pathway)[ora.q_matched.astype(float) < ALPHA])


t0 = time.perf_counter()
BASE_CALLS['clustering'] = clustering_calls(BASE_MODEL)
cached_b = stored(T71, f'method_b_{DS}_{BASE}__null_{NULL_LABEL}')
ora_b = pd.DataFrame({c: cached_b[f'ora|{c}'] for c in cached_b['ora|__columns'].astype(str)})
assert BASE_CALLS['clustering'] == set(ora_b.pathway_id[ora_b.q_matched.astype(float) < ALPHA]), 'G3: clustering calls'
guard_log.append(f"G3: rerunning 71's clustering on the cached model reproduces its {len(BASE_CALLS['clustering'])} calls "
                 f"({time.perf_counter() - t0:.0f} s)")
print('· ' + guard_log[-1])
pd.DataFrame([(m, s) for m, v in BASE_CALLS.items() for s in sorted(v)], columns=['method', 'set']).to_csv(
    tables / 'baseline_calls.csv', index=False)
print({m: len(v) for m, v in BASE_CALLS.items()})

## 3 · Planting runs

Per replicate: choose 15 sets, plant 60% of their members in the group-1 specimens' units, refit the planted genes
(gene model, specimen-shape LR, segment LR) under the cached dispersion, splice, and run the three methods.

In [ ]:
GIDX = pd.Series(np.arange(len(GENES)), index=GENES)
G1_UNITS = GROUP == 1
from pseudospace.pathway_networks import segment_cuts  # noqa: E402
mouse_units = ~np.asarray(D['case'], bool)
SHAPE_CENTRE = float(segment_cuts(POS[mouse_units], np.char.replace(SEG[mouse_units].astype(str), 'PT-', ''))[1])
print(f'Planted logistic centre (mouse S2/S3 cut): {SHAPE_CENTRE:.3f}')


def splice_model(base, planted_idx, refit):
    curves = {n: base.specimen_curves[n].copy() for n in base.specimen_curves}
    for n in curves:
        curves[n][planted_idx] = refit.specimen_curves[n]
    arrays = {}
    for name in ('reference', 'case', 'delta', 'se'):
        a = getattr(base, name).copy()
        a[planted_idx] = getattr(refit, name)
        arrays[name] = a
    return replace(base, specimen_curves=curves, **arrays)


def run_replicate(scenario, amplitude, rep, k=K_SETS, arm=0):
    seed = [SEED, SCENARIOS.index(scenario), int(amplitude * 10), rep] + ([arm] if arm else [])
    rng = np.random.default_rng(seed)
    chosen = choose_sets(SETS, rng, k=k, sizes=PLANT_SIZES, max_jaccard=MAX_JACCARD, frac=FRAC, min_members=MIN_PLANTED)
    signs = {s: float(rng.choice([-1., 1.])) for s in chosen}
    genes = [g for s in chosen for g in chosen[s]]
    gsign = np.array([signs[s] for s in chosen for _ in chosen[s]])
    idx = GIDX[genes].to_numpy()
    f = effect_shape(POS, scenario, G1_UNITS, centre=SHAPE_CENTRE)
    mult = np.ones((len(POS), len(idx)))
    mult[G1_UNITS] = np.exp(np.log(2) * amplitude * f[G1_UNITS, None] * gsign[None, :])
    counts = plant_counts(COUNTS[:, idx].toarray(), mult, rng)
    lib = LIB + counts.sum(axis=1) - np.asarray(COUNTS[:, idx].sum(axis=1)).ravel()
    alpha = ALPHA_ALL[idx]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        refit = standard_gene_model(counts, LIB, POS, GROUP, SPEC, grid=BASE_MODEL.grid, genes=genes, nuisance_shape=NUIS,
                                    dispersion=alpha)
        spec = specimen_shape_lr(counts, LIB, POS, GROUP, SPEC, {'alpha': alpha, 'converged': np.ones(len(idx), bool)},
                                 nuisance_shape=NUIS)
        seg = segment_step_lr(counts, LIB, SEG, GROUP, SPEC, alpha, nuisance=NUIS)
    T, LR = BASE_T.copy(), BASE_LR.copy()
    T[idx], LR[idx] = refit.statistics.T_spatial.to_numpy(float), spec['LR_spec']
    ST, SLR = SEG_T.copy(), SEG_LR.copy()
    ST[idx], SLR[idx] = seg['T_step'], seg['LR_spec']
    calls = {'positional F-test': f_calls(T, LR)[0], 'segment F-test': s_calls(ST, SLR),
             'clustering': clustering_calls(splice_model(BASE_MODEL, idx, refit))}
    out = {'sets': np.array(list(chosen)), 'signs': np.array([signs[s] for s in chosen]), 'genes': np.array(genes),
           'library change': np.array([float(np.median(np.abs(lib / LIB - 1)))])}
    for m, c in calls.items():
        out[f'calls|{m}'] = np.array(sorted(c), dtype=str)
    return out


CODE_KEY = code_digest(run_replicate, choose_sets, effect_shape, plant_counts, splice_model)   # one string key
from pseudospace.stats_gam import gam_internal_knots  # noqa: E402
assert np.allclose(gam_internal_knots(POS, basis_df=6), BASE_MODEL.knots), 'refits must share the cached knots'
t0 = time.perf_counter()
RESULTS = {}
for scenario in SCENARIOS:
    for amplitude in AMPLITUDES:
        for rep in range(N_REP):
            key = (scenario, amplitude, rep)
            RESULTS[key] = cached_payload(f'replicate_{scenario}_{amplitude}_{rep}', lambda s=scenario, a=amplitude, r=rep: run_replicate(s, a, r),
                                          root=cache, params={'logic': NOTEBOOK_LOGIC_VERSION, 'key': list(map(str, key))},
                                          inputs={'genes': len(GENES), 'sets': len(SETS)}, verbose=False,
                                          code=CODE_KEY)
        print(f'  {scenario} A={amplitude}: done ({time.perf_counter() - t0:.0f} s)')

### 3.1 · The low-dose arm (notebook 100's post hoc arm, frozen for this notebook)

Added after the frozen runs showed that 15 planted sets (about 460 genes, 4–6% of the universe) spill into hundreds
of other sets through shared genes (a diagnostic run: unplanted genes' z unchanged, r 0.9999997; removing planted
genes from the other sets cut the calls from 694 to 10). Here 3 sets per replicate (about 90 genes), otherwise
identical, to measure each method at the dose of a single program. Not judged by the frozen rules.

In [ ]:
K_LOW = 3
RESULTS_LOW = {}
for scenario in SCENARIOS:
    for amplitude in AMPLITUDES:
        for rep in range(N_REP):
            key = (scenario, amplitude, rep)
            RESULTS_LOW[key] = cached_payload(f'low_replicate_{scenario}_{amplitude}_{rep}',
                                              lambda s=scenario, a=amplitude, r=rep: run_replicate(s, a, r, k=K_LOW, arm=1),
                                              root=cache, params={'logic': NOTEBOOK_LOGIC_VERSION, 'key': list(map(str, key)), 'k': K_LOW},
                                              inputs={'genes': len(GENES), 'sets': len(SETS)}, verbose=False, code=CODE_KEY)
        print(f'  low dose · {scenario} A={amplitude}: done ({time.perf_counter() - t0:.0f} s)')

## 4 · Summaries and readings

In [ ]:
SET_GENES = {s: set(v) for s, v in SETS.items()}


def score(results, arm):
    rows, reps = [], []
    for (scenario, amplitude, rep), r in results.items():
        planted = set(r['sets'].astype(str))
        pgenes = set(r['genes'].astype(str))
        reps.append({'arm': arm, 'scenario': scenario, 'A': amplitude, 'replicate': rep, 'sets': ';'.join(sorted(planted)),
                     'signs': ';'.join(map(str, r['signs'].astype(int))), 'planted genes': len(pgenes),
                     'median |library change|': float(r['library change'][0])})
        for m in METHODS:
            calls = set(r[f'calls|{m}'].astype(str))
            base = BASE_CALLS[m]
            new_false = (calls - planted) - base
            spill = sum(1 for s in new_false if SET_GENES[s] & pgenes)
            rows.append({'arm': arm, 'method': m, 'scenario': scenario, 'A': amplitude, 'replicate': rep,
                         'sensitivity': len(calls & planted) / len(planted), 'planted called': len(calls & planted),
                         'new false calls': len(new_false), 'spillover': spill, 'clean false calls': len(new_false) - spill,
                         'lost calls': len((base - planted) - calls),
                         'precision': len(calls & planted) / max(len(calls & planted) + len(new_false), 1)})
    return pd.DataFrame(rows), reps


SCORES, REPS = score(RESULTS, 'frozen (15 sets)')
SCORES_LOW, REPS_LOW = score(RESULTS_LOW, 'low dose (3 sets)')
REPS = REPS + REPS_LOW
pd.DataFrame(REPS).to_csv(tables / 'replicates.csv', index=False)
pd.concat([SCORES, SCORES_LOW]).to_csv(tables / 'scores.csv', index=False)
pd.DataFrame(REPS).groupby(['scenario', 'A'])['median |library change|'].max().to_csv(tables / 'library_change.csv')
SUMMARY = SCORES.groupby(['method', 'scenario', 'A']).agg(
    sensitivity=('sensitivity', 'mean'), sens_min=('sensitivity', 'min'), sens_max=('sensitivity', 'max'),
    new_false=('new false calls', 'mean'), spillover=('spillover', 'mean'), lost=('lost calls', 'mean'),
    precision=('precision', 'mean')).reset_index()
SUMMARY.to_csv(tables / 'summary.csv', index=False)
display(SUMMARY.pivot_table(index=['scenario', 'A'], columns='method', values=['sensitivity', 'new_false']).round(2))

READ = []
for scenario in SCENARIOS:
    s1 = SUMMARY[SUMMARY.scenario.eq(scenario) & SUMMARY.A.eq(1.)].set_index('method')
    detects = [m for m in METHODS if s1.loc[m, 'sensitivity'] >= DETECT_SENS and s1.loc[m, 'new_false'] <= DETECT_FALSE]
    if detects:
        top = max(s1.loc[detects, 'sensitivity'])
        best = [m for m in detects if s1.loc[m, 'sensitivity'] >= top - TIE]
    else:
        best = []
    READ.append({'scenario': scenario, **{f'{m} detects': m in detects for m in METHODS},
                 'best (ties within 0.1)': ', '.join(best) if best else 'none',
                 **{f'{m} sensitivity A=1 / 0.5': f"{s1.loc[m, 'sensitivity']:.2f} / "
                    f"{SUMMARY[SUMMARY.scenario.eq(scenario) & SUMMARY.A.eq(.5) & SUMMARY.method.eq(m)].sensitivity.iloc[0]:.2f}"
                    for m in METHODS}})
READ = pd.DataFrame(READ)
READ.to_csv(tables / 'readings.csv', index=False)
display(READ)

# post hoc readings: clean false calls (no planted gene) in place of all new false calls; and the low-dose arm
POST = []
for arm, sc in (('frozen (15 sets)', SCORES), ('low dose (3 sets)', SCORES_LOW)):
    summ = sc.groupby(['method', 'scenario', 'A'])[['sensitivity', 'new false calls', 'spillover', 'clean false calls', 'precision']].mean()
    for (m, scenario, a), r in summ.iterrows():
        POST.append({'arm': arm, 'method': m, 'scenario': scenario, 'A': a, **r.round(3).to_dict(),
                     'detects (sensitivity ≥ 0.5, clean false ≤ 2)': bool(r['sensitivity'] >= DETECT_SENS and r['clean false calls'] <= DETECT_FALSE)})
POST = pd.DataFrame(POST)
POST.to_csv(tables / 'post_hoc_readings.csv', index=False)
display(POST.pivot_table(index=['arm', 'scenario', 'A'], columns='method', values=['sensitivity', 'new false calls', 'clean false calls']).round(2))

### 4.1 · Against notebook 100 (AKI): does each method's profile generalise?

In [ ]:
P100 = pd.read_csv(results_root / 'pt_planted_benchmark' / 'tables' / 'post_hoc_readings.csv')
P100['arm'] = P100.arm.str.replace('post hoc ', '', regex=False)
key_cols = ['arm', 'method', 'scenario', 'A']
CMP = POST.merge(P100, on=key_cols, suffixes=(' hvm', ' AKI'))
det = 'detects (sensitivity ≥ 0.5, clean false ≤ 2)'
CMP['same reading'] = CMP[f'{det} hvm'] == CMP[f'{det} AKI']
CMP.to_csv(tables / 'against_aki.csv', index=False)
GEN = CMP[CMP.A.eq(1.)].groupby(['arm', 'method'])['same reading'].all().rename('generalises (all scenarios, A = 1)')
GEN.to_frame().to_csv(tables / 'generalisation.csv')
display(CMP.pivot_table(index=['arm', 'scenario', 'A'], columns='method', values=['sensitivity hvm', 'sensitivity AKI']).round(2))
display(GEN)

## 5 · Figures

In [ ]:
x = np.linspace(0, 1, 501)
fig, ax = plt.subplots(figsize=(70 * MM, 45 * MM))
for sc, color in zip(SCENARIOS, ['#2a78d6', '#d55e00', INK, '#9C93D3']):
    ax.plot(x, effect_shape(x, sc, np.ones_like(x, bool), centre=SHAPE_CENTRE), color=color, label=sc)
for c in ZONES:
    ax.axvline(c, color=MUTED, lw=.5, ls=':')
ax.axhline(0, color=MUTED, lw=.5)
ax.set_xlabel('PT position')
ax.set_ylabel('planted f(x)')
ax.legend(loc='upper left')
save(fig, '100a_planted_shapes')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(150 * MM, 50 * MM), sharey=True)
for ax, a in zip(axes, AMPLITUDES):
    width = .25
    for j, m in enumerate(METHODS):
        b = SUMMARY[SUMMARY.method.eq(m) & SUMMARY.A.eq(a)].set_index('scenario').reindex(SCENARIOS)
        xs = np.arange(len(SCENARIOS)) + (j - 1) * width
        ax.bar(xs, b.sensitivity, width=width, color=METHOD_COLOR[m], label=m)
        ax.vlines(xs, b.sens_min, b.sens_max, color=INK, lw=.6)
    ax.set_xticks(range(len(SCENARIOS)), SCENARIOS)
    ax.set_title(f'A = {a} log2')
    ax.set_ylim(0, 1.02)
axes[0].set_ylabel('share of planted sets called')
axes[1].legend(loc='upper right')
save(fig, '100b_sensitivity')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(150 * MM, 45 * MM), sharey=True)
for ax, a in zip(axes, AMPLITUDES):
    for j, m in enumerate(METHODS):
        b = SUMMARY[SUMMARY.method.eq(m) & SUMMARY.A.eq(a)].set_index('scenario').reindex(SCENARIOS)
        xs = np.arange(len(SCENARIOS)) + (j - 1) * .25
        ax.bar(xs, b.new_false, width=.25, color=METHOD_COLOR[m], label=m)
    ax.set_xticks(range(len(SCENARIOS)), SCENARIOS)
    ax.set_title(f'A = {a} log2')
axes[0].set_ylabel('new false calls per replicate')
axes[1].legend(loc='upper right')
save(fig, '100c_false_calls')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(150 * MM, 50 * MM))
low = SCORES_LOW.groupby(['method', 'scenario', 'A'])[['sensitivity', 'new false calls']].mean().reset_index()
for ax, col, lab in zip(axes, ('sensitivity', 'new false calls'), ('share of planted sets called', 'new false calls per replicate')):
    for j, m in enumerate(METHODS):
        for k, a in enumerate(AMPLITUDES):
            b = low[low.method.eq(m) & low.A.eq(a)].set_index('scenario').reindex(SCENARIOS)
            xs = np.arange(len(SCENARIOS)) + (j - 1) * .25 + (k - .5) * .1
            ax.bar(xs, b[col], width=.1, color=METHOD_COLOR[m], alpha=.5 if a < 1 else 1., label=f'{m}, A={a}' if col == 'sensitivity' else None)
    ax.set_xticks(range(len(SCENARIOS)), SCENARIOS)
    ax.set_ylabel(lab)
axes[0].legend(loc='upper left', bbox_to_anchor=(0, 1.35), ncol=3)
save(fig, '100d_low_dose')
plt.show()

## 6 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', "Planted genes are refitted with the cached dispersion (71's condition alphas), the same grid and knots, and the library left unchanged in the fits; the library change from planting is reported."),
    ('reading', "The segment F-test baseline is fitted here once on every gene; its moderated F uses the same covariate (log mean count) as the positional F-test."),
    ('reading', "Clustering's fair null is the other relabeling (swap_IR2A2_Ctrl1A2), as notebook 71 ran it."),
    ('reading', 'Replicate seeds are (100, scenario index, 10 × A, replicate); the low-dose arm appends 1.'),
    ('reading', "Notebook 100's code with the human vs mouse base; the low-dose arm and the clean-false reading are frozen here (protocol)."),
    ('reading', "The planted logistic sits at the mouse units' S2/S3 cut (segment_cuts), this dataset's equivalent of notebook 100's 0.65."),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '103_pt_planted_benchmark_hvm', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'baseline calls': {m: len(v) for m, v in BASE_CALLS.items()}, 'replicates': len(RESULTS), 'low-dose replicates': len(RESULTS_LOW),
       'readings': READ.to_dict('records'), 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps(RUN, indent=2, default=str))